Function that will be used to crawl the publication sites and acquire a list of unique publications along with other info

In [ ]:
import requests
from bs4 import BeautifulSoup as bsSoup
import time
import warnings
import sys
import pandas as pd

# ok this is a bad practice but only for now and will remove later
#warnings.filterwarnings("ignore")
#__warningregistry__

In [ ]:
# scrape the links for all the publications of https://sitescrape.awh.durham.ac.uk/comp42315/publicationfull_year_characteranimation.htm
publicationsIndex = "https://sitescrape.awh.durham.ac.uk/comp42315/publicationfull_year_characteranimation.htm"

def getLinks (url: str) -> list :
    sitePrefix = "https://sitescrape.awh.durham.ac.uk/comp42315/"

    if (not isinstance(url, str)) :
        print("The URL needs to be a string!")
        return []

    # wait a little so we don't overload the server
    time.sleep(2)
    publicationsIndex = requests.get(url, verify = False)

    if (publicationsIndex.status_code != 200) :
        print (f"Error; status code returned: {publicationsIndex.status_code}")
        return []

    publicationsIndexSoup = bsSoup(publicationsIndex.content, "html.parser").body

    if (publicationsIndexSoup == None) :
        print ("Nothin to parse on the site")
        return []

    publicationLinksA = publicationsIndexSoup.find("p", class_ = "TextOption").find_all("a")

    if (publicationLinksA == None) :
        print ("No links found")
        return []
    
    links = [sitePrefix + n.get("href") for n in publicationLinksA]

    if (links[0] == None) :
        print("No links found")
        return []

    return links

publicationLinks = [publicationsIndex] + getLinks(publicationsIndex)

def scrapePublications (urls: list) -> list:
    uniquePublications = set()

    for url in urls :
        content = []

        time.sleep(2)
        publicationSite = requests.get(url, verify = False)
        
        if (publicationSite.status_code != 200) :
            print(f"Failed for link {url}, status code: {publicationSite.status_code}, continuing execution for the remaining links")
            continue

        publicationSiteSoup = bsSoup(publicationSite.content, "html.parser").body

        if (publicationSiteSoup == None) :
            print (f"Found nothing to parse on site {url}, continuing execution for the remaining links")
            continue

        publicationsDiv = publicationSiteSoup.find_all("div", style = "margin-left: var(--size-marginleft)")

        if (len(publicationsDiv) == 0) :
            print (f"Couldn't find publications on site {url}, continuing execution for the remaining links")
            continue

        content = [n.find("div", class_ = "w3-container w3-cell w3-mobile w3-cell-middle").text.strip() for n in publicationsDiv]

    # compare the titles somewhere
    # find first by? and this is a cutoff for the title
    return content 

def scrapeImpactScores (urls: list) -> dict :
    impactScores = {}
    # if a site has been seen, ignore it
    seenSites = set()
    n = 0
    
    for url in urls :
        #n += 1
        #print(n)
        uniqueScoresOnThePage = []
        # change that 0 to 1 later
        time.sleep(0)
        url = url.replace("year", "impactfactor")
        
        scoreSite = requests.get(url, verify = False)

        if (scoreSite.status_code != 200) :
            print(f"Failed for link {url}, status code: {scoreSite.status_code}, continuing execution for the remaining links")
            continue

        scoreSiteSoup = bsSoup(scoreSite.content, "html.parser").body

        if (scoreSiteSoup == None) :
            print (f"Found nothing to parse on site {url}, continuing execution for the remaining links")
            continue

        scoreSiteSoup = scoreSiteSoup.find("div", id = "divBackground")
        
        if (scoreSiteSoup == None) :
            print (f"Couldn't find scores on the page {url}, continuing execution for the remaining links")
            continue

        scoreSiteSoupP = scoreSiteSoup.find_all("p", class_ = "TextOption")

        if (scoreSiteSoupP == None) :
            print (f"Couldn't find scores on the page {url}, continuing execution for the remaining links")
            continue

        paragraphWithScores = scoreSiteSoupP [2]

        uniqueScoresOnThePage = [n.text for n in paragraphWithScores.find_all("a")]

        if (len(uniqueScoresOnThePage) == 0) :
            print (f"Couldn't find scores on the page {url}, continuing execution for the remaining links")
            continue

        # for each link in unique links find the publication titles that correspond
        for score in uniqueScoresOnThePage :
            currentH2 = scoreSiteSoup.find("h2", id = score)

            if (currentH2 == None) :
                # no publications under this score
                continue

            if (score not in impactScores) :
                impactScores [score] = []

            div = currentH2.findNext()

            publicationsWithThisScore = div.find_all("div", class_ = "w3-cell-row")

            if (publicationsWithThisScore == None) :
                # no publications under this score
                continue

            for publication in publicationsWithThisScore :
                publicationText = publication.find("div", class_ = "w3-container w3-cell w3-mobile w3-cell-middle")

                # I'd need to find the title here
                title = publicationText.text.split("by")[0].strip()

                if (title in seenSites) :
                    continue

                seenSites.add(title)
                impactScores[score].append(publicationText.text)
                #print(title)

    return impactScores
                                                      
#print (publicationLinks)
#print(scrapePublications(publicationLinks))   
scoreDict = scrapeImpactScores(publicationLinks)
print(scoreDict)

This information should then be stored and displayed in an appropriate format. Displayed to the user should be: publication title, publication venue, year and author list of every unique publication record on the target website.

In [ ]:
# extract info from score dict
# create a new dictionary with cleaned data title: [[authors], year, publication venue]
publications = {}
publicationsData : pd.DataFrame
s = 0

for k in scoreDict :
    for n in scoreDict[k]:
        # 110 (started indexing at 0) unique publications it seems
        remainder = ""
        initialClean = n.translate({ord(i): None for i in '\t\r\n'})
        # remove the "Webpage" at the end
        initialClean_2 = initialClean[:-8]

        # ok, it's important that we only get the first occurance of by
        splitAt = initialClean_2.find("by")
        title = initialClean_2[:splitAt].strip()
        remainder = initialClean_2[splitAt + 3:]

        # works to around there, needs improving this bit
        # as expected there are "in" in the names so that does not work
        splitAt = remainder.find(" in ")
        authors = remainder[:splitAt]
        year = remainder[splitAt + 4:splitAt + 8]
        publicationVenue = remainder[splitAt + 8:]

        #print (s)
        #s += 1
        #print (n)
        #print ("---")
        # print (title)
        # print (authors)
        # print (year)
        # print (publicationVenue)
        # print (k)

        # splitting authors into a list of names
        authorList1 = authors.split(",")
        authorList2 = authorList1[-1].split("and")
        #print("---")
        authorList1Strip = [n.strip() for n in authorList1]
        authorList2Strip = [n.strip() for n in authorList2]
        authorList = authorList1Strip[:-1] + authorList2Strip
        #print (authorList)

        # add to dictionary
        publications[f"{title}"] = [authorList, year, publicationVenue]

# also create a pandas dataframe from it
authorsForDf = []
for k in publications :
    authors : str = ""

    for i in range(0, len(publications[k][0])) :
        print (publications[k][0][i] + ", ")
        if (i == len(publications[k][0]) - 1) :
            authorsForDf += publications[k][0][i]
        else :
            authors += publications[k][0][i] + ", "

    authorsForDf.append(authors)   

#print (authorsForDf)

_data = {"title" : [k for k in publications], "authors" : [i for i in publications[k][0] for k in publications], "year" : [publications[k][1] for k in publications], "publication venue" : [publications[k][2] for k in publications]}

# print (len(_data["publication venue"]))
# print (_data["authors"])



#publicationsData = pd.DataFrame(data = _data)

# print (publications)  
# for k in publications :
#     for i in publications[k][0] :
#         print (i)   

#print ([n for sublist in [[i for i in publications[k][0]] for k in publications] for n in sublist])
    

In [ ]:
# print to the console
# export as a text file
# export as a csv

# print to the console
def printScrapedDataToConsole () -> None :
    for k in publications :
        print (f"Title: {k}")
        print ("---")
        sys.stdout.write("Authors: ")
        for n in publications[k][0] :
            sys.stdout.write(n)
            # make a mechanism for adding commas, "and" and a dot
        print (f"\nYear published: {publications[k][1]}")
        print (f"Publication venue: {publications[k][2]}\n")

# exporting as a csv
#def exportToCsv (fileName: str, dataToExport: dict) :

printScrapedDataToConsole()
    

The records should be manipulatable: at minimum you should be able to display sorted information according to descending year values, descending number of author values, the titles from A to Z, and finally by the impact of the papers.

Question 2: <br>
a. Selection of training set. Write code splitting the given dataset into the training and testing sets. The output of the code should be a dataset consisting of the rows of drone.csv that have been chosen. In the text part (part f) of your submission, please justify your choice.

In [ ]:
import random as rd
import matplotlib as mplt
from sklearn.feature_selection import r_regression, SelectKBest

droneData = pd.read_csv("drone.csv")

In [ ]:
def takeBootstrapSample (inBagProportion: float, df: pd.DataFrame) -> tuple :
    rowIndicies = []
    for i in range(1,droneData.shape[0] + 1) :
        rowIndicies.append(i)

    trainingSubsetRows = rd.sample(rowIndicies, round(len(rowIndicies) * inBagProportion))
    # converting to sets
    allRowsS = set (rowIndicies)
    trainingRowsS = set (trainingSubsetRows)
    validationSubsetRowsS = allRowsS - trainingRowsS
    # now get the actual rows from the dataframe
    trainingRowsBool = []
    validationRowsBool = []
    for i in rowIndicies :
        if i in trainingRowsS :
            trainingRowsBool.append(True)
        else :
            trainingRowsBool.append(False)

        if i in validationSubsetRowsS :
            validationRowsBool.append(True)
        else :
            validationRowsBool.append(False)

    trainingData = df.iloc[trainingRowsBool]
    validationData = df.iloc[validationRowsBool]

    return (trainingData, validationData)

bootstrapSample = takeBootstrapSample(0.75, droneData)


b. Feature selection (applied to the training set only). In this task you are requested to write code for testing of two feature selection approaches of your choice. The output of implementation of each approach should be the dataset with three features obtained from the output of part a) by removal of those columns that have not been selected as features.

In [ ]:
trainingDataset = bootstrapSample[0]

# this returns the number of NaNs for each column 
#print (trainingDataset.isna().sum())
# this returns a specific row that contains both NaNs: 3107
#print(trainingDataset[trainingDataset["load"].isna()])
# I can just remove it as it is a single row in more than 4000 observations
trainingDatasetRemNAs = trainingDataset.drop(3107)

# split the dataframe into the predictors and response
allPredictors = trainingDatasetRemNAs.loc[:, trainingDataset.columns != "warning"]
response = trainingDatasetRemNAs["warning"]

# convert to numpy array
allPredictorsNp = allPredictors.to_numpy()
responseNp = response.to_numpy()

# ok, so this is one approach - scores via r regression and select k best

# calculate the r-regression scores
scores = r_regression(allPredictorsNp, responseNp)

scores
# hmm

# select k best approach
selectedPredictors_kBest = SelectKBest(r_regression, k = 3).fit_transform(allPredictorsNp, responseNp)

# select via L1 penalisation
#lasso = 

